# Correlation Analysis: Logical Error Distribution vs Fixed-Class Decoding LLRs

This notebook analyzes the correlation between:
1. **Logical error probability** (from pre-computed distribution)
2. **Correction weight (LLR)** from fixed-class decoding

**Hypothesis**: If the distribution-based gap proxy methods work as intended, logical errors with higher probability should produce lower LLRs when used for fixed-class decoding.

## 1. Run Data Collection Script

First, run the data collection script from the terminal. This separates the long-running simulation from the interactive analysis.

```bash
# Navigate to project root
cd /path/to/ldpc-post-selection

# Run with parallel processing (recommended)
python simulations/analysis/scripts/run_distribution_correlation_analysis.py \
    --n-qubits 144 \
    --p 0.003 \
    --n-shots 1000 \
    --n-errors 10 \
    --n-jobs 126 \
    --output simulations/data/correlation_analysis/bb_n144_p0.003_shots1k.parquet
```

**Options:**
- `--n-qubits, -n`: BB code variant (72, 90, 108, 144, 288, 360, 756)
- `--p`: Physical error rate
- `--n-shots`: Number of shots to analyze (each requires 11 decoder calls)
- `--n-errors`: Number of representative errors to sample from distribution
- `--n-jobs, -j`: Number of parallel jobs (`-1` for all CPUs, default: `1`)
- `--output, -o`: Output path for results (parquet file)

Run `python simulations/analysis/scripts/run_distribution_correlation_analysis.py --help` for all options.

In [15]:
# Autoreload extension for Jupyter notebooks
%load_ext autoreload
%autoreload 2

# Standard library imports
import json
import os
import sys
from pathlib import Path

# Third-party library imports
import numpy as np
import pandas as pd
from scipy import stats

# Plotly for interactive visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add the parent directory to sys.path
sys.path.append(str(Path(os.getcwd()).parent.parent))

# Pandas display settings
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.6f}'.format)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Load Results

In [16]:
# ==============================================================================
# CONFIGURATION - Point to your results file
# ==============================================================================

RESULTS_PATH = Path("../../data/correlation_analysis/bb_n144_p0.003_shots1k.parquet")

# ==============================================================================
# Load results and metadata
# ==============================================================================

if not RESULTS_PATH.exists():
    print(f"Results file not found: {RESULTS_PATH.resolve()}")
    print("\nPlease run the data collection script first:")
    print("")
    print("  python simulations/analysis/scripts/run_distribution_correlation_analysis.py \\")
    print("      --n-qubits 144 --p 0.003 --n-shots 10000 --n-jobs -1 \\")
    print(f"      --output {RESULTS_PATH}")
    raise FileNotFoundError(f"Results not found at {RESULTS_PATH}")

# Load DataFrame
df_results = pd.read_parquet(RESULTS_PATH)

# Load metadata
metadata_path = RESULTS_PATH.with_suffix(".json")
if metadata_path.exists():
    with open(metadata_path) as f:
        metadata = json.load(f)
else:
    metadata = {}

# Display summary
print(f"Loaded {len(df_results):,} records from {RESULTS_PATH.name}")
print(f"\nMetadata:")
for key, value in metadata.items():
    if isinstance(value, list) and len(value) > 5:
        print(f"  {key}: [{value[0]}, {value[1]}, ..., {value[-1]}] (len={len(value)})")
    elif isinstance(value, dict):
        print(f"  {key}: {value}")
    else:
        print(f"  {key}: {value}")

print(f"\nDataFrame preview:")
df_results.head(10)

Loaded 10,000 records from bb_n144_p0.003_shots1k.parquet

Metadata:
  n_qubits: 144
  distance: 12
  p: 0.003
  n_shots: 1000
  n_representative_errors: 10
  num_observables: 12
  logical_error_rate: 0.009543945312500024
  distribution_shots: 4096000
  decoder_params: {'max_iter': 30, 'bp_method': 'minimum_sum', 'lsd_method': 'LSD_0', 'lsd_order': 0}
  seed: 42
  n_jobs: 126
  selected_error_indices: [4, 3862, ..., 4060] (len=10)
  selected_error_ranks: [0, 454, ..., 4094] (len=10)
  selected_error_probs: [0.004988232886524097, 0.00038371022204031517, ..., 0.0] (len=10)

DataFrame preview:


,shot_id,error_rank,error_index,error_prob,fixed_llr,best_llr,llr_delta
0,0,0,4,0.004988,315.114223,131.747453,183.366770
1,0,454,3862,0.000384,427.080822,131.747453,295.333369
2,0,909,305,0.000205,421.318035,131.747453,289.570582
3,0,1364,2038,0.000153,416.086842,131.747453,284.339389
4,0,1819,3241,0.000128,621.739575,131.747453,489.992122
5,0,2274,3495,0.000102,592.927688,131.747453,461.180235
6,0,2729,3564,0.000077,459.657221,131.747453,327.909768
7,0,3184,3327,0.000051,541.127860,131.747453,409.380408
8,0,3639,3306,0.000026,363.923228,131.747453,232.175775
9,0,4094,4060,0.000000,446.173248,131.747453,314.425795


## 3. Statistical Analysis

In [17]:
# Compute correlations
print("=" * 60)
print("STATISTICAL ANALYSIS")
print("=" * 60)

# 1. Spearman rank correlation: error_rank vs fixed_llr
spearman_corr, spearman_p = stats.spearmanr(df_results["error_rank"], df_results["fixed_llr"])
print(f"\n1. Spearman Rank Correlation (error_rank vs fixed_llr):")
print(f"   Correlation: {spearman_corr:.4f}")
print(f"   p-value: {spearman_p:.2e}")
print(f"   Interpretation: {'Significant' if spearman_p < 0.05 else 'Not significant'} at alpha=0.05")
if spearman_corr > 0:
    print(f"   Direction: Positive (higher rank = higher LLR, as expected)")
else:
    print(f"   Direction: Negative (unexpected!)")

# 2. Pearson correlation: log(error_prob) vs fixed_llr
# Filter out zero probabilities for log
df_nonzero = df_results[df_results["error_prob"] > 0].copy()
df_nonzero["log_error_prob"] = np.log10(df_nonzero["error_prob"])
pearson_corr, pearson_p = stats.pearsonr(df_nonzero["log_error_prob"], df_nonzero["fixed_llr"])
print(f"\n2. Pearson Correlation (log10(error_prob) vs fixed_llr):")
print(f"   Correlation: {pearson_corr:.4f}")
print(f"   p-value: {pearson_p:.2e}")
print(f"   Interpretation: {'Significant' if pearson_p < 0.05 else 'Not significant'} at alpha=0.05")
if pearson_corr < 0:
    print(f"   Direction: Negative (higher prob = lower LLR, as expected)")
else:
    print(f"   Direction: Positive (unexpected!)")

# 3. Kruskal-Wallis test: Are LLR distributions different across rank groups?
groups = [group["fixed_llr"].values for _, group in df_results.groupby("error_rank")]
kruskal_stat, kruskal_p = stats.kruskal(*groups)
print(f"\n3. Kruskal-Wallis Test (LLR distributions across rank groups):")
print(f"   H-statistic: {kruskal_stat:.4f}")
print(f"   p-value: {kruskal_p:.2e}")
print(f"   Interpretation: {'Distributions differ significantly' if kruskal_p < 0.05 else 'No significant difference'}")

# 4. Summary statistics per error rank
print(f"\n4. Summary Statistics by Error Rank:")
summary = df_results.groupby("error_rank").agg({
    "error_prob": "first",
    "fixed_llr": ["mean", "std", "median"],
    "llr_delta": ["mean", "std"],
}).round(4)
summary.columns = ["error_prob", "llr_mean", "llr_std", "llr_median", "delta_mean", "delta_std"]
print(summary.to_string())

STATISTICAL ANALYSIS

1. Spearman Rank Correlation (error_rank vs fixed_llr):
   Correlation: 0.0009
   p-value: 9.31e-01
   Interpretation: Not significant at alpha=0.05
   Direction: Positive (higher rank = higher LLR, as expected)

2. Pearson Correlation (log10(error_prob) vs fixed_llr):
   Correlation: 0.0330
   p-value: 1.76e-03
   Interpretation: Significant at alpha=0.05
   Direction: Positive (unexpected!)

3. Kruskal-Wallis Test (LLR distributions across rank groups):
   H-statistic: 40.7769
   p-value: 5.49e-06
   Interpretation: Distributions differ significantly

4. Summary Statistics by Error Rank:
            error_prob   llr_mean    llr_std  llr_median  delta_mean  delta_std
error_rank                                                                     
0             0.005000 481.642100 159.297700  450.987200  296.323500 164.826400
454           0.000400 482.717100 105.737200  465.877800  297.398500 108.022800
909           0.000200 485.702300 114.034500  467.689300  300

## 4. Per-Shot Correlation Analysis

The aggregated statistics above treat all shots as interchangeable. However, what matters for gap proxy selection is the **within-shot correlation**: does the distribution rank predict which error has lower LLR *for a given syndrome*?

For each shot, we compute:
1. **Per-shot Spearman correlation** between rank and LLR (10 data points per shot)
2. **Optimal selection rate**: Does rank-0 error have the lowest LLR?
3. **Regret**: How much LLR penalty for using rank-0 vs the true minimum?

In [18]:
# ==============================================================================
# Per-Shot Correlation Analysis
# ==============================================================================

# Compute per-shot statistics
per_shot_stats = []

for shot_id, group in df_results.groupby("shot_id"):
    # Sort by error rank to ensure consistent ordering
    group = group.sort_values("error_rank")
    
    # Per-shot Spearman correlation (rank vs LLR)
    rho, p_val = stats.spearmanr(group["error_rank"], group["fixed_llr"])
    
    # Find which rank achieved the minimum LLR
    min_idx = group["fixed_llr"].idxmin()
    min_llr_rank = group.loc[min_idx, "error_rank"]
    min_llr = group["fixed_llr"].min()
    
    # Rank-0 statistics
    rank0_llr = group[group["error_rank"] == 0]["fixed_llr"].values[0]
    rank0_is_best = (min_llr_rank == 0)
    
    # Regret: LLR penalty for using rank-0 instead of optimal
    regret = rank0_llr - min_llr
    
    # Baseline difficulty (best_llr from standard decoding)
    best_llr = group["best_llr"].iloc[0]
    
    # Mean LLR for random selection baseline
    mean_llr = group["fixed_llr"].mean()
    random_regret = mean_llr - min_llr
    
    per_shot_stats.append({
        "shot_id": shot_id,
        "spearman_rho": rho,
        "spearman_p": p_val,
        "min_llr_rank": min_llr_rank,
        "min_llr": min_llr,
        "rank0_llr": rank0_llr,
        "rank0_is_best": rank0_is_best,
        "regret": regret,
        "random_regret": random_regret,
        "best_llr": best_llr,
        "mean_fixed_llr": mean_llr,
    })

df_per_shot = pd.DataFrame(per_shot_stats)

print("=" * 60)
print("PER-SHOT CORRELATION ANALYSIS")
print("=" * 60)
print(f"\nAnalyzed {len(df_per_shot)} shots")
print(f"\nDataFrame preview:")
df_per_shot.head(10)

PER-SHOT CORRELATION ANALYSIS

Analyzed 1000 shots

DataFrame preview:


,shot_id,spearman_rho,spearman_p,min_llr_rank,min_llr,rank0_llr,rank0_is_best,regret,random_regret,best_llr,mean_fixed_llr
0,0,0.333333,0.346594,0,315.114223,315.114223,True,0.000000,145.400651,131.747453,460.514874
1,1,-0.018182,0.960240,1819,356.150979,506.900153,False,150.749174,129.176583,146.402040,485.327562
2,2,-0.284848,0.425038,3639,300.322923,521.221799,False,220.898876,177.742170,138.755619,478.065093
3,3,0.575758,0.081553,1819,282.382828,349.000720,False,66.617892,153.990047,114.304215,436.372875
4,4,0.042424,0.907364,3184,308.656160,343.487330,False,34.831169,146.325722,178.269834,454.981882
5,5,0.175758,0.627188,909,388.359343,431.592206,False,43.232863,110.346444,235.492123,498.705787
6,6,-0.224242,0.533401,0,354.510552,354.510552,True,0.000000,110.092031,146.740005,464.602583
7,7,0.115152,0.751420,0,400.766613,400.766613,True,0.000000,54.426540,142.712375,455.193153
8,8,0.357576,0.310376,0,301.742057,301.742057,True,0.000000,102.129366,185.575477,403.871423
9,9,0.054545,0.881036,2729,324.337268,382.014743,False,57.677475,176.080107,159.675629,500.417375


In [19]:
# ==============================================================================
# Per-Shot Correlation Statistics
# ==============================================================================

print("=" * 60)
print("PER-SHOT SPEARMAN CORRELATION STATISTICS")
print("=" * 60)

# Filter out NaN correlations (can happen if all LLRs are identical in a shot)
valid_correlations = df_per_shot["spearman_rho"].dropna()
n_valid = len(valid_correlations)
n_total = len(df_per_shot)

print(f"\nValid correlations: {n_valid}/{n_total} shots")

# Basic statistics
mean_rho = valid_correlations.mean()
std_rho = valid_correlations.std()
median_rho = valid_correlations.median()

print(f"\nPer-shot Spearman correlation (rank vs LLR):")
print(f"  Mean:   {mean_rho:.4f}")
print(f"  Std:    {std_rho:.4f}")
print(f"  Median: {median_rho:.4f}")
print(f"  Min:    {valid_correlations.min():.4f}")
print(f"  Max:    {valid_correlations.max():.4f}")

# One-sample t-test: is mean correlation significantly different from 0?
t_stat, t_pval = stats.ttest_1samp(valid_correlations, 0)
print(f"\nOne-sample t-test (H0: mean = 0):")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {t_pval:.2e}")
print(f"  Interpretation: {'Mean is significantly different from 0' if t_pval < 0.05 else 'Mean is NOT significantly different from 0'}")

# Wilcoxon signed-rank test (non-parametric alternative)
w_stat, w_pval = stats.wilcoxon(valid_correlations)
print(f"\nWilcoxon signed-rank test (H0: median = 0):")
print(f"  W-statistic: {w_stat:.1f}")
print(f"  p-value: {w_pval:.2e}")

# Fraction of positive/negative correlations
n_positive = (valid_correlations > 0).sum()
n_negative = (valid_correlations < 0).sum()
n_zero = (valid_correlations == 0).sum()
print(f"\nCorrelation sign distribution:")
print(f"  Positive (rank ↑ → LLR ↑): {n_positive} ({100*n_positive/n_valid:.1f}%)")
print(f"  Negative (rank ↑ → LLR ↓): {n_negative} ({100*n_negative/n_valid:.1f}%)")
print(f"  Zero:                       {n_zero} ({100*n_zero/n_valid:.1f}%)")

PER-SHOT SPEARMAN CORRELATION STATISTICS

Valid correlations: 1000/1000 shots

Per-shot Spearman correlation (rank vs LLR):
  Mean:   0.0050
  Std:    0.3510
  Median: 0.0061
  Min:    -0.8545
  Max:    0.8667

One-sample t-test (H0: mean = 0):
  t-statistic: 0.4499
  p-value: 6.53e-01
  Interpretation: Mean is NOT significantly different from 0

Wilcoxon signed-rank test (H0: median = 0):
  W-statistic: 245046.0
  p-value: 5.69e-01

Correlation sign distribution:
  Positive (rank ↑ → LLR ↑): 506 (50.6%)
  Negative (rank ↑ → LLR ↓): 494 (49.4%)
  Zero:                       0 (0.0%)


In [20]:
# ==============================================================================
# Histogram of Per-Shot Correlations
# ==============================================================================

fig = go.Figure()

# Histogram of per-shot Spearman correlations
fig.add_trace(go.Histogram(
    x=valid_correlations,
    nbinsx=50,
    name="Per-shot ρ",
    marker_color="steelblue",
    opacity=0.7,
))

# Add vertical line at mean
fig.add_vline(
    x=mean_rho,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Mean = {mean_rho:.3f}",
    annotation_position="top",
)

# Add vertical line at 0
fig.add_vline(
    x=0,
    line_dash="solid",
    line_color="black",
    line_width=1,
)

fig.update_layout(
    title="Distribution of Per-Shot Spearman Correlations (Rank vs LLR)",
    xaxis_title="Spearman ρ (per shot)",
    yaxis_title="Count",
    showlegend=False,
)

# Add annotation with statistics
fig.add_annotation(
    text=f"Mean = {mean_rho:.3f}<br>Std = {std_rho:.3f}<br>t-test p = {t_pval:.2e}",
    xref="paper", yref="paper",
    x=0.02, y=0.98,
    showarrow=False,
    font=dict(size=11),
    bgcolor="white",
    bordercolor="gray",
    borderwidth=1,
    align="left",
)

fig.show()

In [21]:
# ==============================================================================
# Optimal Selection Rate and Regret Analysis
# ==============================================================================

print("=" * 60)
print("OPTIMAL SELECTION RATE")
print("=" * 60)

n_errors = len(df_results["error_rank"].unique())
random_baseline = 1.0 / n_errors

# Rank-0 selection success rate
rank0_success_rate = df_per_shot["rank0_is_best"].mean()
print(f"\nIf we always select rank-0 (highest probability) error:")
print(f"  Success rate (rank-0 has min LLR): {100*rank0_success_rate:.2f}%")
print(f"  Random baseline:                   {100*random_baseline:.2f}%")
print(f"  Improvement over random:           {100*(rank0_success_rate - random_baseline):.2f}pp")

# Binomial test: is rank-0 success rate better than random?
binom_result = stats.binomtest(
    k=int(df_per_shot["rank0_is_best"].sum()),
    n=len(df_per_shot),
    p=random_baseline,
    alternative="greater",
)
print(f"\nBinomial test (H0: success rate = random):")
print(f"  p-value: {binom_result.pvalue:.2e}")
print(f"  95% CI: [{binom_result.proportion_ci(confidence_level=0.95).low:.4f}, {binom_result.proportion_ci(confidence_level=0.95).high:.4f}]")

print("\n" + "=" * 60)
print("REGRET ANALYSIS")
print("=" * 60)

mean_regret = df_per_shot["regret"].mean()
mean_random_regret = df_per_shot["random_regret"].mean()

print(f"\nRegret = LLR(selected) - LLR(optimal)")
print(f"\nRank-0 selection regret:")
print(f"  Mean:   {mean_regret:.2f}")
print(f"  Median: {df_per_shot['regret'].median():.2f}")
print(f"  Std:    {df_per_shot['regret'].std():.2f}")

print(f"\nRandom selection regret (expected):")
print(f"  Mean:   {mean_random_regret:.2f}")
print(f"  Median: {df_per_shot['random_regret'].median():.2f}")
print(f"  Std:    {df_per_shot['random_regret'].std():.2f}")

print(f"\nRegret reduction (random → rank-0): {mean_random_regret - mean_regret:.2f}")

# Paired t-test: is rank-0 regret lower than random regret?
t_regret, p_regret = stats.ttest_rel(df_per_shot["regret"], df_per_shot["random_regret"])
print(f"\nPaired t-test (H0: rank-0 regret = random regret):")
print(f"  t-statistic: {t_regret:.4f}")
print(f"  p-value: {p_regret:.2e}")

OPTIMAL SELECTION RATE

If we always select rank-0 (highest probability) error:
  Success rate (rank-0 has min LLR): 24.40%
  Random baseline:                   10.00%
  Improvement over random:           14.40pp

Binomial test (H0: success rate = random):
  p-value: 2.35e-39
  95% CI: [0.2217, 1.0000]

REGRET ANALYSIS

Regret = LLR(selected) - LLR(optimal)

Rank-0 selection regret:
  Mean:   125.55
  Median: 77.41
  Std:    149.43

Random selection regret (expected):
  Mean:   121.03
  Median: 115.38
  Std:    44.30

Regret reduction (random → rank-0): -4.52

Paired t-test (H0: rank-0 regret = random regret):
  t-statistic: 1.0198
  p-value: 3.08e-01


In [22]:
# ==============================================================================
# Rank-Stratified Success Rate (Top-k Analysis)
# ==============================================================================

print("=" * 60)
print("RANK-STRATIFIED SUCCESS RATE")
print("=" * 60)
print("\nIf we pick among top-k errors by distribution probability,")
print("how often does the true minimum LLR fall within that set?")
print()

unique_ranks = sorted(df_results["error_rank"].unique())
n_total_ranks = len(unique_ranks)

topk_results = []
for k in range(1, n_total_ranks + 1):
    top_k_ranks = set(unique_ranks[:k])
    success_count = (df_per_shot["min_llr_rank"].isin(top_k_ranks)).sum()
    success_rate = success_count / len(df_per_shot)
    random_rate = k / n_total_ranks
    topk_results.append({
        "k": k,
        "success_rate": success_rate,
        "random_rate": random_rate,
        "improvement": success_rate - random_rate,
    })

df_topk = pd.DataFrame(topk_results)
print(df_topk.to_string(index=False, formatters={
    "success_rate": "{:.2%}".format,
    "random_rate": "{:.2%}".format,
    "improvement": "{:+.2%}".format,
}))

RANK-STRATIFIED SUCCESS RATE

If we pick among top-k errors by distribution probability,
how often does the true minimum LLR fall within that set?

 k success_rate random_rate improvement
 1       24.40%      10.00%     +14.40%
 2       32.00%      20.00%     +12.00%
 3       41.40%      30.00%     +11.40%
 4       50.10%      40.00%     +10.10%
 5       55.90%      50.00%      +5.90%
 6       64.40%      60.00%      +4.40%
 7       72.40%      70.00%      +2.40%
 8       82.10%      80.00%      +2.10%
 9       90.90%      90.00%      +0.90%
10      100.00%     100.00%      +0.00%


In [23]:
# ==============================================================================
# Conditional Analysis by Shot Difficulty
# ==============================================================================

print("=" * 60)
print("CONDITIONAL ANALYSIS BY SHOT DIFFICULTY")
print("=" * 60)
print("\nDoes correlation vary with shot difficulty (baseline LLR)?")

# Stratify by best_llr quartiles
df_per_shot["difficulty_quartile"] = pd.qcut(
    df_per_shot["best_llr"], 
    q=4, 
    labels=["Q1 (easy)", "Q2", "Q3", "Q4 (hard)"]
)

# Compute statistics per quartile
quartile_stats = df_per_shot.groupby("difficulty_quartile", observed=True).agg({
    "spearman_rho": ["mean", "std", "count"],
    "rank0_is_best": "mean",
    "regret": "mean",
    "best_llr": ["min", "max"],
}).round(4)

quartile_stats.columns = [
    "rho_mean", "rho_std", "n_shots",
    "rank0_success", "mean_regret",
    "llr_min", "llr_max"
]

print("\nPer-quartile statistics:")
print(quartile_stats.to_string())

# Test if correlation varies significantly across quartiles
groups_by_quartile = [
    group["spearman_rho"].dropna() 
    for _, group in df_per_shot.groupby("difficulty_quartile", observed=True)
]
kw_stat, kw_pval = stats.kruskal(*groups_by_quartile)
print(f"\nKruskal-Wallis test (correlation differs across difficulty quartiles):")
print(f"  H-statistic: {kw_stat:.4f}")
print(f"  p-value: {kw_pval:.2e}")

CONDITIONAL ANALYSIS BY SHOT DIFFICULTY

Does correlation vary with shot difficulty (baseline LLR)?

Per-quartile statistics:
                     rho_mean  rho_std  n_shots  rank0_success  mean_regret    llr_min    llr_max
difficulty_quartile                                                                              
Q1 (easy)           -0.027200 0.369200      250       0.196000   170.400000  87.024600 159.866000
Q2                  -0.014900 0.360100      250       0.252000   119.398300 159.942300 181.089000
Q3                   0.031900 0.350700      250       0.280000   111.783100 181.119800 208.834700
Q4 (hard)            0.030300 0.320100      250       0.248000   100.633700 208.838900 422.319400

Kruskal-Wallis test (correlation differs across difficulty quartiles):
  H-statistic: 4.4067
  p-value: 2.21e-01


In [24]:
# ==============================================================================
# Visualization: Per-Shot Correlation vs Shot Difficulty
# ==============================================================================

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Per-Shot Correlation vs Baseline LLR",
        "Rank-0 Success Rate by Difficulty Quartile"
    ),
)

# Left: Scatter plot of correlation vs baseline LLR
fig.add_trace(
    go.Scatter(
        x=df_per_shot["best_llr"],
        y=df_per_shot["spearman_rho"],
        mode="markers",
        marker=dict(size=4, opacity=0.3, color="steelblue"),
        name="Shots",
        hovertemplate="Baseline LLR: %{x:.1f}<br>Correlation: %{y:.3f}<extra></extra>",
    ),
    row=1, col=1,
)

# Add horizontal line at y=0
fig.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=1)

# Right: Bar plot of rank-0 success rate by quartile
quartile_labels = quartile_stats.index.tolist()
success_rates = quartile_stats["rank0_success"].values
random_line = 1.0 / n_errors

fig.add_trace(
    go.Bar(
        x=quartile_labels,
        y=success_rates,
        name="Rank-0 Success",
        marker_color="steelblue",
        text=[f"{r:.1%}" for r in success_rates],
        textposition="outside",
    ),
    row=1, col=2,
)

# Add random baseline
fig.add_hline(
    y=random_line, 
    line_dash="dash", 
    line_color="red", 
    row=1, col=2,
    annotation_text=f"Random: {random_line:.1%}",
    annotation_position="top right",
)

fig.update_xaxes(title_text="Baseline LLR (shot difficulty)", row=1, col=1)
fig.update_yaxes(title_text="Per-Shot Spearman ρ", row=1, col=1)
fig.update_xaxes(title_text="Difficulty Quartile", row=1, col=2)
fig.update_yaxes(title_text="Rank-0 Success Rate", row=1, col=2)

fig.update_layout(
    height=400,
    showlegend=False,
    title_text="Per-Shot Analysis by Shot Difficulty",
)

fig.show()

## 5. Aggregated Visualizations

These visualizations show the aggregated relationship between error rank and LLR across all shots.

In [25]:
# Box plot: fixed_llr by error_rank
fig = px.box(
    df_results,
    x="error_rank",
    y="fixed_llr",
    title="Fixed-Class LLR Distribution by Error Rank",
    labels={
        "error_rank": "Error Rank (0=most likely, higher=less likely)",
        "fixed_llr": "Fixed-Class LLR",
    },
)
fig.update_layout(
    xaxis_title="Error Rank (0=most likely)",
    yaxis_title="Fixed-Class LLR",
    showlegend=False,
)
fig.add_annotation(
    text=f"Spearman r = {spearman_corr:.3f} (p = {spearman_p:.2e})",
    xref="paper", yref="paper",
    x=0.02, y=0.98,
    showarrow=False,
    font=dict(size=12),
    bgcolor="white",
)
fig.show()

In [26]:
# Scatter plot: error_prob vs mean fixed_llr with error bars
summary_for_plot = df_results.groupby(["error_rank", "error_prob"]).agg({
    "fixed_llr": ["mean", "std", "count"]
}).reset_index()
summary_for_plot.columns = ["error_rank", "error_prob", "llr_mean", "llr_std", "count"]
summary_for_plot["llr_sem"] = summary_for_plot["llr_std"] / np.sqrt(summary_for_plot["count"])

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=summary_for_plot["error_prob"],
    y=summary_for_plot["llr_mean"],
    error_y=dict(
        type="data",
        array=summary_for_plot["llr_sem"] * 1.96,  # 95% CI
        visible=True,
    ),
    mode="markers",
    marker=dict(size=10),
    text=[f"Rank {r}" for r in summary_for_plot["error_rank"]],
    hovertemplate="Prob: %{x:.2e}<br>Mean LLR: %{y:.2f}<br>%{text}<extra></extra>",
))

fig.update_layout(
    title="Error Probability vs Mean Fixed-Class LLR",
    xaxis_title="Error Probability (from distribution)",
    yaxis_title="Mean Fixed-Class LLR",
    xaxis_type="log",
)
fig.add_annotation(
    text=f"Pearson r = {pearson_corr:.3f} (p = {pearson_p:.2e})",
    xref="paper", yref="paper",
    x=0.02, y=0.98,
    showarrow=False,
    font=dict(size=12),
    bgcolor="white",
)
fig.show()

In [27]:
# Heatmap: shot-level LLRs (subset of shots x errors)
n_shots_total = df_results["shot_id"].nunique()
n_shots_to_show = min(100, n_shots_total)  # Show first 100 shots
df_subset = df_results[df_results["shot_id"] < n_shots_to_show]

pivot_df = df_subset.pivot(index="shot_id", columns="error_rank", values="fixed_llr")

fig = px.imshow(
    pivot_df.values,
    labels=dict(x="Error Rank", y="Shot ID", color="Fixed-Class LLR"),
    x=[f"Rank {r}" for r in pivot_df.columns],
    y=pivot_df.index,
    title=f"Fixed-Class LLR Heatmap (first {n_shots_to_show} shots)",
    color_continuous_scale="Viridis",
    aspect="auto",
)
fig.update_layout(
    xaxis_title="Error Rank (0=most likely)",
    yaxis_title="Shot ID",
)
fig.show()

In [28]:
# CDF comparison: high-prob errors vs low-prob errors
# Split into top 3 ranks (high prob) vs bottom 3 ranks (low prob)
unique_ranks = sorted(df_results["error_rank"].unique())
n_ranks = len(unique_ranks)
top_ranks = unique_ranks[:3]
bottom_ranks = unique_ranks[-3:]

df_top = df_results[df_results["error_rank"].isin(top_ranks)]
df_bottom = df_results[df_results["error_rank"].isin(bottom_ranks)]

fig = go.Figure()

# Top ranks (high probability errors)
sorted_top = np.sort(df_top["fixed_llr"])
cdf_top = np.arange(1, len(sorted_top) + 1) / len(sorted_top)
fig.add_trace(go.Scatter(
    x=sorted_top,
    y=cdf_top,
    mode="lines",
    name=f"High-prob errors (ranks {list(top_ranks)})",
    line=dict(color="blue"),
))

# Bottom ranks (low probability errors)
sorted_bottom = np.sort(df_bottom["fixed_llr"])
cdf_bottom = np.arange(1, len(sorted_bottom) + 1) / len(sorted_bottom)
fig.add_trace(go.Scatter(
    x=sorted_bottom,
    y=cdf_bottom,
    mode="lines",
    name=f"Low-prob errors (ranks {list(bottom_ranks)})",
    line=dict(color="red"),
))

# Mann-Whitney U test
mw_stat, mw_p = stats.mannwhitneyu(df_top["fixed_llr"], df_bottom["fixed_llr"], alternative="less")

fig.update_layout(
    title="CDF Comparison: High-Probability vs Low-Probability Errors",
    xaxis_title="Fixed-Class LLR",
    yaxis_title="Cumulative Probability",
    legend=dict(x=0.6, y=0.1),
)
fig.add_annotation(
    text=f"Mann-Whitney U: p = {mw_p:.2e}",
    xref="paper", yref="paper",
    x=0.98, y=0.5,
    showarrow=False,
    font=dict(size=12),
    bgcolor="white",
)
fig.show()

print(f"\nMann-Whitney U Test (high-prob < low-prob):")
print(f"  U-statistic: {mw_stat:.1f}")
print(f"  p-value: {mw_p:.2e}")
print(f"  Interpretation: {'High-prob errors have significantly lower LLRs' if mw_p < 0.05 else 'No significant difference'}")


Mann-Whitney U Test (high-prob < low-prob):
  U-statistic: 4578670.0
  p-value: 8.80e-01
  Interpretation: No significant difference


In [29]:
# Violin plot for more detailed distribution view
fig = px.violin(
    df_results,
    x="error_rank",
    y="fixed_llr",
    box=True,
    points="outliers",
    title="Fixed-Class LLR Distribution by Error Rank (Violin Plot)",
    labels={
        "error_rank": "Error Rank (0=most likely)",
        "fixed_llr": "Fixed-Class LLR",
    },
)
fig.show()

## 6. Conclusions

In [30]:
print("=" * 60)
print("CONCLUSIONS")
print("=" * 60)

# Aggregated analysis summary
print("\n--- Aggregated Analysis ---")
print(f"Spearman Correlation (rank vs LLR): r = {spearman_corr:.4f}")
if abs(spearman_corr) < 0.1:
    agg_correlation_strength = "negligible"
elif abs(spearman_corr) < 0.3:
    agg_correlation_strength = "weak"
elif abs(spearman_corr) < 0.5:
    agg_correlation_strength = "moderate"
else:
    agg_correlation_strength = "strong"
print(f"  Correlation strength: {agg_correlation_strength}")

# Per-shot analysis summary
print("\n--- Per-Shot Analysis ---")
print(f"Mean per-shot Spearman ρ: {mean_rho:.4f} ± {std_rho:.4f}")
print(f"  Significantly different from 0: {'Yes' if t_pval < 0.05 else 'No'} (p = {t_pval:.2e})")
print(f"\nRank-0 selection success rate: {100*rank0_success_rate:.2f}%")
print(f"  Random baseline: {100*random_baseline:.2f}%")
print(f"  Improvement: {100*(rank0_success_rate - random_baseline):+.2f}pp")
print(f"\nMean regret (rank-0): {mean_regret:.2f}")
print(f"Mean regret (random): {mean_random_regret:.2f}")

print("\n" + "=" * 60)
print("IMPLICATIONS FOR GAP PROXY METHODS")
print("=" * 60)

# Determine if distribution-based selection is useful
if mean_rho < 0.1 and rank0_success_rate < random_baseline * 1.5:
    print("""
The per-shot analysis confirms WEAK within-shot correlation between the
logical error distribution and fixed-class decoding LLRs.

Key findings:
1. Per-shot Spearman correlations are centered around 0
2. Rank-0 selection success rate is close to random baseline
3. Distribution-based selection provides minimal regret reduction

This explains why distribution-based gap proxy methods (most-likely-first,
weighted-random) do not outperform random sampling: the pre-computed
distribution does not predict which logical class will have lowest LLR
for a specific syndrome.

Possible explanations:
1. LLRs are highly shot-dependent (determined by syndrome structure)
2. The global distribution captures error frequency, not per-shot competitiveness
3. Fixed-class decoding success depends on syndrome-specific factors
""")
elif mean_rho > 0.1 or rank0_success_rate > random_baseline * 1.5:
    print("""
The per-shot analysis shows SOME correlation between the logical error
distribution and fixed-class decoding LLRs, but it may be too weak to
provide significant improvement over random sampling.

The distribution-based methods may provide marginal benefit, but the
high variance in per-shot correlations limits their effectiveness.
""")

CONCLUSIONS

--- Aggregated Analysis ---
Spearman Correlation (rank vs LLR): r = 0.0009
  Correlation strength: negligible

--- Per-Shot Analysis ---
Mean per-shot Spearman ρ: 0.0050 ± 0.3510
  Significantly different from 0: No (p = 6.53e-01)

Rank-0 selection success rate: 24.40%
  Random baseline: 10.00%
  Improvement: +14.40pp

Mean regret (rank-0): 125.55
Mean regret (random): 121.03

IMPLICATIONS FOR GAP PROXY METHODS

The per-shot analysis shows SOME correlation between the logical error
distribution and fixed-class decoding LLRs, but it may be too weak to
provide significant improvement over random sampling.

The distribution-based methods may provide marginal benefit, but the
high variance in per-shot correlations limits their effectiveness.

